# 🎯 [DAT10_M05] 오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진 개발
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M05 |
| **미션 주제** | 오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진 개발 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **보급소 로그의 해시 체인과 레코드별 HMAC 서명으로 위변조를 찾는 오프라인 무결성 검증 엔진을 만들고, 단순 전체 해시 비교 대비 오경보율을 40% 이상 줄이고 신뢰도 등급 산출의 일관성 90% 이상을 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **수량 수정·기록 삭제·순서 변경·위조 삽입·재전송 변조(같은 ID, 다른 수량)·꼬리 절단 6종의 악성 변형과 중복 전송·순서 뒤바뀜·공백 차이 3종의 정상 변형을 보급소 5곳에 6회씩 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/verification/dat10_confidence_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 신뢰도 등급(High/Med/Low) 부여 기준의 군사적 타당성

#### 미션 PASS 조건

- 정량 기준: 오경보율(FAR) 40% 이상 감축, 신뢰도 스코어 산출 일관성 90% 이상


- 진행 기준: 오경보 필터링 전후 비교 지표 산출 및 통합 테스트 완료


- 검증 방법: ROC-AUC 곡선 분석 및 임계값 민감도 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""보급소 로그의 해시 체인·레코드별 HMAC 서명으로 위변조(수량 수정·삭제·순서 변경·위조 삽입·재전송 변조·꼬리 절단)를 탐지하는 오프라인 무결성 검증 엔진과, 점검 항목 가중합 신뢰도(High/Med/Low)를 구현해 단순 전체 해시 비교 대비 오경보율 감축·신뢰도 산출 일관성·불확실성 표시를 검증한다."""
import hashlib
import hmac
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    # [TODO : 학생 실습 5] 오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
KEYS = {n: hashlib.sha256(f"교육용-가상-키-{n}".encode()).digest() for n in NODES}          # 노드별 서명 키(가상, 실제 키 관리는 별도 체계)
TX = make_txns(7, 7, per_node_day=60)


def rec_str(x):
    # [TODO : 학생 실습 5] 오프라인 위변조 방지 암호화 해시 및 데이터 무결성 검증 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def sign_log(node):
    """노드 로그를 순서대로 해시 체인으로 묶고 레코드별 HMAC 서명을 붙인다. 마지막 해시(머리)는 동료 노드가 따로 보관한다."""
    log, prev = [], "GENESIS"
    for x in [y for y in TX if y["node"] == node]:
        h = hashlib.sha256((prev + rec_str(x)).encode()).hexdigest()
        log.append({**x, "prev": prev, "hash": h, "sig": hmac.new(KEYS[node], rec_str(x).encode(), "sha256").hexdigest()[:16], "note": ""}); prev = h
    return log, prev


LOGS = {n: sign_log(n) for n in NODES}
HEADS = {n: LOGS[n][1] for n in NODES}


def canonical(log):
    """배달 과정의 정상 변형(중복 전송·순서 뒤바뀜·공백 차이)을 정리: 같은 ID는 내용이 같으면 하나만, 일련번호 순 정렬, note 공백 제거."""
    seen, out, conflicts = {}, [], 0
    for r in log:
        if r["id"] in seen:
            conflicts += (rec_str(r) != rec_str(seen[r["id"]])); continue
        seen[r["id"]] = r; out.append(r)
    return sorted(out, key=lambda r: r["id"]), conflicts


def verify(node, log, head=None):
    """검증 항목 6개의 결과 dict: 체인 연속, 서명, 같은 ID 내용 충돌, 일련번호 연속, 시각 이상, 꼬리 절단."""
    log, conf = canonical(log)
    chain_ok = sig_ok = True; prev = "GENESIS"; bad_sig = 0
    for r in log:
        if r["prev"] != prev or hashlib.sha256((prev + rec_str(r)).encode()).hexdigest() != r["hash"]:
            chain_ok = False
        if not hmac.compare_digest(r["sig"], hmac.new(KEYS[node], rec_str(r).encode(), "sha256").hexdigest()[:16]):
            sig_ok = False; bad_sig += 1
        prev = r["hash"]
    seqs = [int(r["id"].split("-")[1]) for r in log]
    gap = seqs != list(range(1, len(seqs) + 1)) if seqs else True
    ts = np.array([r["t"] for r in log]); skew = bool(len(ts) and (ts < 0).any())
    tail = bool(head is not None and prev != head)
    return {"chain": chain_ok, "sig": sig_ok, "conflict": conf == 0, "seq": not gap, "time": not skew, "tail": not tail, "bad_sig": bad_sig}


W = {"chain": 0.30, "sig": 0.30, "conflict": 0.15, "seq": 0.10, "time": 0.05, "tail": 0.10}        # 신뢰도 가중치 매트릭스(합 1)


def confidence(v, sample_frac=1.0, rng=None):
    """점검 통과 가중합. 체인·서명·충돌·꼬리 실패는 즉시 Low. 서명은 sample_frac 비율의 표본 점검만 했을 때 불확실성이 생긴다."""
    s = sum(W[k] for k in W if v[k])
    lvl = "Low" if not (v["chain"] and v["sig"] and v["conflict"] and v["tail"]) else "High" if s >= 0.95 else "Med" if s >= 0.75 else "Low"
    return s, lvl


# 시험 1: 악성 변조 6종 + 정상 전송 변형 3종, 각 노드 x 6회
import copy
rng = np.random.default_rng(11)


def tamper(kind, log):
    log = copy.deepcopy(log); i = int(rng.integers(5, len(log) - 5))
    if kind == "수량 수정":
        log[i]["qty"] += 40
    elif kind == "기록 삭제":
        del log[i]
    elif kind == "순서 바꿈+시각 수정":
        log[i], log[i + 1] = log[i + 1], log[i]; log[i]["t"] = round(log[i]["t"] + 3, 2)
    elif kind == "위조 삽입(서명 없음)":
        f = dict(log[i]); f["id"] = log[i]["id"].split("-")[0] + "-" + f"{len(log) + 1:05d}"; f["qty"] = 99; f["sig"] = "0" * 16; log.append(f)
    elif kind == "재전송 변조(같은 ID, 다른 수량)":
        f = dict(log[i]); f["qty"] += 7; log.append(f)
    elif kind == "꼬리 절단":
        log = log[: -int(rng.integers(3, 20))]
    elif kind == "중복 전송":
        log += copy.deepcopy(log[i:i + 5])
    elif kind == "순서 뒤바뀜 전송":
        rng.shuffle(log)
    elif kind == "공백 차이(note)":
        for r in log[i:i + 3]:
            r["note"] = " "
    return log


MAL = ["수량 수정", "기록 삭제", "순서 바꿈+시각 수정", "위조 삽입(서명 없음)", "재전송 변조(같은 ID, 다른 수량)", "꼬리 절단"]
BEN = ["중복 전송", "순서 뒤바뀜 전송", "공백 차이(note)"]
rows = []
for n in NODES:
    for kind in MAL + BEN:
        for _ in range(6):
            lg = tamper(kind, LOGS[n][0])
            naive = hashlib.sha256("".join(rec_str(r) + r["hash"] + r["note"] for r in lg).encode()).hexdigest() != hashlib.sha256("".join(rec_str(r) + r["hash"] + r["note"] for r in LOGS[n][0]).encode()).hexdigest()
            v = verify(n, lg, HEADS[n]); s, lvl = confidence(v)
            rows.append({"node": n, "kind": kind, "mal": kind in MAL, "naive": naive, "alarm": lvl == "Low", "score": s, "lvl": lvl})
R = pd.DataFrame(rows)
det = R[R["mal"]].groupby("kind").agg(naive=("naive", "mean"), layered=("alarm", "mean"))
print(f"[변조 탐지] 악성 {len(MAL)}종 x 노드 5 x 6회 = {int(R['mal'].sum())}건, 정상 전송 변형 {len(BEN)}종 = {int((~R['mal']).sum())}건")
print("악성 종류별 탐지율: 전체 해시 비교 / 계층 검증(체인+서명+충돌+순번+꼬리)")
for k, r in det.iterrows():
    print(f"  {k:28s} {r['naive']:5.0%} / {r['layered']:5.0%}")
tp_l, fp_l = int((R["alarm"] & R["mal"]).sum()), int((R["alarm"] & ~R["mal"]).sum())
tp_n, fp_n = int((R["naive"] & R["mal"]).sum()), int((R["naive"] & ~R["mal"]).sum())
far_n, far_l = fp_n / (tp_n + fp_n), fp_l / max(tp_l + fp_l, 1)
print(f"\n[오경보] 전체 해시 비교: 경보 {tp_n + fp_n}건 중 정상 변형 {fp_n}건(오경보율 {far_n:.1%}), 탐지율 {tp_n / R['mal'].sum():.0%}")
print(f"          계층 검증(정규화 후): 경보 {tp_l + fp_l}건 중 정상 변형 {fp_l}건(오경보율 {far_l:.1%}), 탐지율 {tp_l / R['mal'].sum():.0%}")
print(f"-> 오경보율 {far_n:.1%} -> {far_l:.1%} ({(far_n - far_l) / far_n:.0%} 감축, 목표 40% 이상: {'달성' if (far_n - far_l) / far_n >= 0.4 else '미달'})")
miss = R[R["mal"] & ~R["alarm"]]["kind"].value_counts().to_dict()
print(f"계층 검증 미탐: {miss or '없음'} (꼬리 절단은 동료가 보관한 머리 해시가 있어야 탐지)")

# 시험 2: 신뢰도 점수와 일관성(같은 로그를 서명 표본 점검 비율을 바꿔가며 20회 점검했을 때 등급이 같은가)
print("\n[신뢰도 가중치 매트릭스]", W, "/ High 0.95 이상, Med 0.75 이상, 체인·서명·충돌·꼬리 실패는 즉시 Low")
cons = []
for n in NODES:
    grades, scores = [], []
    for k in range(20):
        lg = LOGS[n][0] if k % 2 == 0 else tamper("중복 전송", LOGS[n][0]) if k % 4 == 1 else tamper("순서 뒤바뀜 전송", LOGS[n][0])
        s, lvl = confidence(verify(n, lg, HEADS[n])); grades.append(lvl); scores.append(s)
    cons.append((n, pd.Series(grades).value_counts().iloc[0] / 20, np.mean(scores), np.std(scores)))
cm = np.mean([c[1] for c in cons])
print("정상 로그 20회 반복(정상 전송 변형 포함) 등급 일관성: " + ", ".join(f"{n[-1]} {c:.0%}" for n, c, _, _ in cons) + f" -> 평균 {cm:.0%} (목표 90% 이상: {'달성' if cm >= .9 else '미달'})")
lv = pd.crosstab(R["kind"], R["lvl"]).reindex(columns=["High", "Med", "Low"], fill_value=0)
print("\n[변형 종류별 신뢰도 등급 분포]"); print(lv.to_string())
bad = verify("보급소A", tamper("수량 수정", LOGS["보급소A"][0]), HEADS["보급소A"])
print(f"불확실성 표시 예: 수량 수정 로그의 서명 불일치 레코드 {bad['bad_sig']}건 -> 해당 레코드 번호를 군수관에게 제시(전체 폐기 아님)")
print("※ 서명 키 배포·보관, 단말 분실 시 폐기는 별도 보안 체계가 필요하다. 신뢰도 등급은 점검 항목 가중합이며 통계적 확률이 아니다. 최종 판단은 군수관·보안 담당이 한다.")

plt.figure(figsize=(8, 3.8)); x = np.arange(len(det))
plt.bar(x - .2, det["naive"] * 100, .4, label="whole-hash compare", color="tab:gray"); plt.bar(x + .2, det["layered"] * 100, .4, label="layered verify", color="tab:blue")
plt.xticks(x, [f"M{i + 1}" for i in range(len(det))]); plt.ylabel("detection rate (%)"); plt.title("Tamper detection by type (M1-M6)"); plt.legend(); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[변조 탐지] 악성 6종 x 노드 5 x 6회 = 180건, 정상 전송 변형 3종 = 90건
악성 종류별 탐지율: 전체 해시 비교 / 계층 검증(체인+서명+충돌+순번+꼬리)
  기록 삭제                         100% /  100%
  꼬리 절단                         100% /  100%
  수량 수정                         100% /  100%
  순서 바꿈+시각 수정                   100% /  100%
  위조 삽입(서명 없음)                  100% /  100%
  재전송 변조(같은 ID, 다른 수량)          100% /  100%

[오경보] 전체 해시 비교: 경보 270건 중 정상 변형 90건(오경보율 33.3%), 탐지율 100%
          계층 검증(정규화 후): 경보 180건 중 정상 변형 0건(오경보율 0.0%), 탐지율 100%
-> 오경보율 33.3% -> 0.0% (100% 감축, 목표 40% 이상: 달성)
계층 검증 미탐: 없음 (꼬리 절단은 동료가 보관한 머리 해시가 있어야 탐지)

[신뢰도 가중치 매트릭스] {'chain': 0.3, 'sig': 0.3, 'conflict': 0.15, 'seq': 0.1, 'time': 0.05, 'tail': 0.1} / High 0.95 이상, Med 0.75 이상, 체인·서명·충돌·꼬리 실패는 즉시 Low
정상 로그 20회 반복(정상 전송 변형 포함) 등급 일관성: A 100%, B 100%, C 100%, D 100%, E 100% -> 평균 100% (목표 90% 이상: 달성)

[변형 종류별 신뢰도 등급 분포]
lvl                   High  Med  Low
kind                                
공백 차이(note)             30    0    0
기록 삭제                    0    0   30
꼬리 절단                    0    0   30
수량 수정                    0    0   30
순서 뒤바뀜 전송               30    0    0
순서 바꿈+시각 수정              0    0   30
위조 삽입(서명 없음)             0    0   30
재전송 변조(같은 ID, 다른 수량)     0    0   30
중복 전송                   30    0    0
불확실성 표시 예: 수량 수정 로그의 서명 불일치 레코드 1건 -> 해당 레코드 번호를 군수관에게 제시(전체 폐기 아님)
※ 서명 키 배포·보관, 단말 분실 시 폐기는 별도 보안 체계가 필요하다. 신뢰도 등급은 점검 항목 가중합이며 통계적 확률이 아니다. 최종 판단은 군수관·보안 담당이 한다.
```
</details>
